# Do dado sintético ao modelo em produção 🤖

## Como usar este caderno?

Este caderno guia você por um fluxo completo de MLOps em escala reduzida:

1. gerar dados sintéticos
2. treinar um modelo
3. publicar o artefato no Hugging Face Hub
4. integrar esse modelo à API construída na semana anterior

A proposta aqui não é apenas "fazer funcionar", mas entender **o papel de cada etapa no ciclo de vida de um modelo em produção**.

Leia cada seção, estude o código de referência e implemente os exercícios no seu ambiente local. Os gabaritos estão nas células colapsadas logo após cada exercício — tente resolver antes de abrir.

---

## O que é essencial, recomendado e desafio?

Ao longo do caderno, as atividades aparecem com três níveis:

* **Essencial**: o mínimo que você precisa conseguir fazer para completar a etapa
* **Recomendado**: amplia a qualidade da solução
* **Desafio**: aprofunda a solução do ponto de vista de engenharia

Isso existe para evitar uma sensação comum em cadernos práticos: achar que tudo tem o mesmo peso. Não tem.

---

## Pré-requisitos

Antes de começar, confirme que você já tem:

* Python funcionando no seu ambiente
* possibilidade de instalar bibliotecas com `pip`
* a API da semana anterior disponível localmente
* conta no Hugging Face
* token de acesso com permissão de escrita no Hugging Face

### Instalação dos pacotes

In [ ]:
# Execute esta célula para instalar os pacotes necessários
!pip install scikit-learn joblib pandas numpy huggingface_hub fastapi uvicorn

### Token do Hugging Face

Você também precisará de uma conta no [Hugging Face](https://huggingface.co) e de um token de acesso com permissão de escrita.

Crie o seu em: **Settings → Access Tokens → New token (role: write)**

> **Atenção:** não comite o token no GitHub e não o deixe hardcoded em arquivos do projeto.

---

## Estrutura esperada do projeto

Antes de chegar ao Bloco 4, seu projeto deve ficar aproximadamente assim:

```text
meu-projeto/
├── main.py
├── model_utils.py
├── routers/
│   └── predict.py
├── model.pkl
├── README.md
└── requirements.txt
```

Essa estrutura não é a única possível, mas ajuda a manter separadas as responsabilidades:

* `model_utils.py` → carregar modelo
* `routers/predict.py` → rotas de ML
* `model.pkl` → artefato treinado
* `README.md` → model card
* `requirements.txt` → dependências do artefato

---

## O contexto

Seu time escolheu um domínio de negócio — **churn de clientes**, **detecção de fraude** ou **risco de crédito**. A API já existe. O que falta é um modelo por trás do `POST /predict`.

A solução desta etapa será gerar dados sintéticos com estrutura realista o suficiente para treinar e validar o pipeline completo.

> **Convenção deste caderno:** os exemplos e alguns gabaritos usarão **fraude** como domínio padrão. Se o seu time escolheu churn ou crédito, adapte nomes de features, interpretação de métricas e exemplos.

Ao final deste caderno, você terá:

* um dataset sintético com features nomeadas e coerentes com seu domínio
* um modelo treinado e serializado
* o modelo publicado no Hugging Face Hub com versionamento
* uma função `load_model(force_download)` pronta para usar na API
* um endpoint `/predict` funcional e testável

---

# BLOCO 1 — Dados sintéticos com sabor de domínio

**Objetivo:** gerar dados que pareçam reais o suficiente para o pipeline funcionar, sem perder tempo modelando distribuições complexas.

---

## Conceitos-chave do Bloco 1

### Por que dados sintéticos?

Para desenvolver e testar um pipeline de ML, você precisa de dados que:

* tenham a estrutura do problema real: features, target e proporção de classes
* sejam reproduzíveis: mesma seed → mesmo dataset
* possam ser gerados em qualquer volume

`make_classification` do scikit-learn faz exatamente isso.

### Parâmetros mais importantes

| Parâmetro       | O que controla                                                 |
| --------------- | -------------------------------------------------------------- |
| `n_samples`     | Quantidade de linhas                                           |
| `n_features`    | Número total de features                                       |
| `n_informative` | Features que realmente importam para a predição                |
| `n_redundant`   | Features correlacionadas com as informativas                   |
| `class_sep`     | Separabilidade das classes (maior = mais fácil de classificar) |
| `weights`       | Proporção de cada classe                                       |
| `random_state`  | Seed para reprodutibilidade                                    |

### O risco dos dados "fáceis demais"

Um `class_sep=3.0` gera um dataset onde qualquer modelo acerta 99%. Isso é inútil para avaliar se seu pipeline funciona de verdade.

Use valores entre `0.8` e `1.5` para simular um problema mais realista.

---

## Exercício 1.1 — Seu primeiro dataset sintético

**Nível:** Essencial  
**Conceito:** `make_classification`, exploração básica, reprodutibilidade

### Referência

In [ ]:
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    class_sep=1.0,
    weights=[0.7, 0.3],
    random_state=42
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y

print(df.head())
print(f"\nDistribuição do target:\n{df['target'].value_counts()}")

### Sua tarefa

Gere um dataset com as configurações acima. Em seguida, experimente alterar `class_sep` para `0.5` e depois para `3.0`.

Observe o que acontece e registre uma observação curta:

* o problema ficou mais fácil ou mais difícil?
* isso mudou a distribuição do target ou apenas a separação entre as classes?

Para refletir, escreva também 2 ou 3 frases respondendo:

> Por que a reprodutibilidade (`random_state`) é importante em MLOps?
> O que acontece se cada execução gerar dados diferentes?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* um `DataFrame` com 6 colunas
* uma coluna `target`
* contagens de classe próximas de 70% / 30%

<details>
<summary>💡 Gabarito</summary>

```python
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    class_sep=1.0,
    weights=[0.7, 0.3],
    random_state=42
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y

print(df.head())
print(f"\nDistribuição: {df['target'].value_counts().to_dict()}")

# class_sep=0.5 torna o problema mais difícil porque as classes se misturam mais.
# class_sep=3.0 torna o problema muito fácil porque as classes ficam bem separadas.
# A distribuição do target tende a continuar parecida; o que muda principalmente
# é a separação entre as classes no espaço das features.
#
# Em MLOps, reprodutibilidade é essencial para comparar experimentos de forma justa.
# Sem random_state fixo, você não sabe se um modelo melhorou de verdade
# ou se apenas recebeu dados mais fáceis.
```

</details>

---

## Exercício 1.2 — Adicionando sabor de domínio

**Nível:** Essencial  
**Conceito:** features nomeadas, distribuições intencionais com NumPy

No exercício anterior, as features se chamavam `feature_0`, `feature_1` etc. Isso serve para testar código, mas dificulta a comunicação com o time e a criação de testes significativos.

Features nomeadas tornam o problema mais compreensível desde o início.

### Referência — domínio de churn

In [ ]:
import numpy as np
import pandas as pd

def gerar_churn(n_samples: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    churn = rng.integers(0, 2, size=n_samples)

    dias_sem_login    = np.where(churn, rng.integers(30, 180, n_samples),
                                        rng.integers(1, 30, n_samples))
    num_chamados      = np.where(churn, rng.integers(3, 15, n_samples),
                                        rng.integers(0, 4, n_samples))
    valor_mensalidade = rng.uniform(50, 500, n_samples).round(2)
    meses_contrato    = np.where(churn, rng.integers(1, 12, n_samples),
                                        rng.integers(6, 60, n_samples))
    nps_score         = np.where(churn, rng.integers(0, 6, n_samples),
                                        rng.integers(5, 11, n_samples))

    return pd.DataFrame({
        "dias_sem_login":    dias_sem_login,
        "num_chamados":      num_chamados,
        "valor_mensalidade": valor_mensalidade,
        "meses_contrato":    meses_contrato,
        "nps_score":         nps_score,
        "churn":             churn
    })

df = gerar_churn(n_samples=2000, seed=42)
print(df.describe())

### Sua tarefa

Escolha o domínio do seu time — **churn**, **fraude** ou **crédito** — e implemente uma função `gerar_dataset(n_samples, seed)` com pelo menos 5 features nomeadas que façam sentido para o domínio.

As features não precisam ser perfeitas. Elas precisam ser **coerentes com a intuição do negócio**.

### Ideias

* **Fraude:** `valor_transacao`, `hora_transacao`, `distancia_ultima_compra`, `tentativas_senha`, `pais_diferente`
* **Crédito:** `renda_mensal`, `divida_atual`, `historico_pagamentos`, `idade`, `num_dependentes`

Além do código, escreva 2 frases explicando por que suas features fazem sentido para o domínio.

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* um `DataFrame` com features nomeadas
* uma coluna target coerente com o domínio
* valores plausíveis para cada coluna
* diferenças médias perceptíveis entre classe 0 e classe 1

<details>
<summary>💡 Gabarito — domínio de fraude</summary>

```python
import numpy as np
import pandas as pd

def gerar_fraude(n_samples: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    fraude = rng.integers(0, 2, size=n_samples)

    valor_transacao = np.where(
        fraude,
        rng.uniform(500, 10000, n_samples),
        rng.uniform(10, 800, n_samples)
    ).round(2)

    hora_transacao = np.where(
        fraude,
        rng.integers(0, 6, n_samples),
        rng.integers(7, 23, n_samples)
    )

    distancia_ultima_compra = np.where(
        fraude,
        rng.uniform(100, 5000, n_samples),
        rng.uniform(0, 50, n_samples)
    ).round(1)

    tentativas_senha = np.where(
        fraude,
        rng.integers(2, 10, n_samples),
        rng.integers(1, 2, n_samples)
    )

    pais_diferente = np.where(
        fraude,
        rng.integers(0, 2, n_samples),
        rng.integers(0, 2, n_samples, p=[0.95, 0.05])
    )

    return pd.DataFrame({
        "valor_transacao":         valor_transacao,
        "hora_transacao":          hora_transacao,
        "distancia_ultima_compra": distancia_ultima_compra,
        "tentativas_senha":        tentativas_senha,
        "pais_diferente":          pais_diferente,
        "fraude":                  fraude
    })

df = gerar_fraude(n_samples=2000, seed=42)
print(df.groupby("fraude").mean().round(2))
```

</details>

---

## Exercício 1.3 — Desafio: função geradora robusta 🎯

**Nível:** Desafio  
**Conceito:** interface limpa, validação de parâmetros, docstring

Sua função geradora vai ser importada pelo código de treinamento, pelo notebook de análise e eventualmente por testes automatizados. Vale a pena deixá-la bem definida.

### Sua tarefa

Refatore a função anterior para:

1. aceitar `proporcao_positivos: float = 0.3`
2. levantar `ValueError` se `proporcao_positivos` não estiver entre `0.05` e `0.95`
3. retornar `(df, X, y)`
4. incluir docstring com parâmetros, retorno e exemplo de uso

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve conseguir:

* chamar a função com diferentes proporções de positivos
* receber erro quando o parâmetro estiver fora do intervalo
* usar `X` e `y` diretamente com sklearn

<details>
<summary>💡 Gabarito</summary>

```python
import numpy as np
import pandas as pd
from typing import Tuple

def gerar_dataset(
    n_samples: int = 1000,
    seed: int = 42,
    proporcao_positivos: float = 0.3
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Gera dataset sintético de detecção de fraude.

    Parâmetros
    ----------
    n_samples : int
        Número de amostras a gerar.
    seed : int
        Seed para reprodutibilidade.
    proporcao_positivos : float
        Proporção da classe positiva. Deve estar entre 0.05 e 0.95.

    Retorna
    -------
    df : pd.DataFrame
        Dataset completo com features e target.
    X : np.ndarray
        Matriz de features.
    y : np.ndarray
        Vetor de targets.

    Exemplo
    -------
    >>> df, X, y = gerar_dataset(n_samples=500, seed=0)
    >>> df.shape
    (500, 6)
    """
    if not (0.05 <= proporcao_positivos <= 0.95):
        raise ValueError(
            f"proporcao_positivos deve estar entre 0.05 e 0.95, "
            f"recebido: {proporcao_positivos}"
        )

    rng = np.random.default_rng(seed)
    fraude = rng.choice(
        [0, 1],
        size=n_samples,
        p=[1 - proporcao_positivos, proporcao_positivos]
    )

    valor_transacao = np.where(
        fraude,
        rng.uniform(500, 10000, n_samples),
        rng.uniform(10, 800, n_samples)
    ).round(2)

    hora_transacao = np.where(
        fraude,
        rng.integers(0, 6, n_samples),
        rng.integers(7, 23, n_samples)
    )

    distancia_ultima_compra = np.where(
        fraude,
        rng.uniform(100, 5000, n_samples),
        rng.uniform(0, 50, n_samples)
    ).round(1)

    tentativas_senha = np.where(
        fraude,
        rng.integers(2, 10, n_samples),
        rng.integers(1, 2, n_samples)
    )

    pais_diferente = (rng.random(n_samples) < np.where(
        fraude, 0.4, 0.05
    )).astype(int)

    df = pd.DataFrame({
        "valor_transacao":         valor_transacao,
        "hora_transacao":          hora_transacao,
        "distancia_ultima_compra": distancia_ultima_compra,
        "tentativas_senha":        tentativas_senha,
        "pais_diferente":          pais_diferente,
        "target":                  fraude
    })

    X = df.drop(columns=["target"]).values
    y = df["target"].values
    return df, X, y
```

</details>

---

# BLOCO 2 — Treinar e serializar o modelo

**Objetivo:** treinar um modelo de classificação, avaliá-lo brevemente e gerar o artefato `.pkl` que será publicado no Hugging Face Hub.

---

## Conceitos-chave do Bloco 2

### O modelo como artefato

Em MLOps, o modelo treinado não é código. Ele é um **artefato**: algo produzido por um processo, versionado, armazenado e consumido por outro processo.

Isso implica que:

* código de treinamento e código de inferência são separados
* o artefato precisa ser reproduzível
* atualizar o modelo não deve exigir mudança no código da API

### Por que `joblib` e não `pickle` diretamente?

`joblib` usa pickle internamente, mas é otimizado para arrays NumPy grandes. Para modelos sklearn, ele é uma escolha comum e recomendada.

### Métricas: além da acurácia

Em problemas desbalanceados, acurácia pode enganar.

Um modelo que sempre prediz "não fraude" pode ter alta acurácia e ainda ser inútil.

| Métrica   | O que mede                               | Quando priorizar                  |
| --------- | ---------------------------------------- | --------------------------------- |
| Acurácia  | % de predições corretas                  | Classes balanceadas               |
| Precision | % dos positivos preditos que são reais   | Custo alto de falso positivo      |
| Recall    | % dos positivos reais detectados         | Custo alto de falso negativo      |
| F1        | Média harmônica entre precision e recall | Quando é preciso equilibrar ambos |

---

## Exercício 2.1 — Treinar e avaliar

**Nível:** Essencial  
**Conceito:** `train_test_split`, `RandomForestClassifier`, métricas básicas

### Referência

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df, X, y = gerar_dataset(n_samples=2000, seed=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

### Sua tarefa

Treine o modelo com seus dados do Bloco 1.

Depois, escreva 2 a 4 frases respondendo:

* a classe positiva teve **precision** maior ou **recall** maior?
* o que isso significa no contexto do seu domínio?
* esse comportamento seria desejável para o negócio?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* `X_train`, `X_test`, `y_train`, `y_test`
* um modelo treinado
* um `classification_report`
* uma interpretação curta das métricas da classe positiva

<details>
<summary>💡 Gabarito</summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df, X, y = gerar_dataset(n_samples=2000, seed=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["legítimo", "fraude"]))

# Recall alto para fraude significa que o modelo detecta a maioria das fraudes reais.
# Precision alta para fraude significa que, quando ele acusa fraude, geralmente está certo.
# Em muitos contextos de fraude, recall costuma ser mais crítico, porque deixar
# passar uma fraude real pode ser mais custoso do que barrar uma transação legítima.
```

</details>

---

## Exercício 2.2 — Serializar o artefato

**Nível:** Essencial  
**Conceito:** `joblib.dump`, ciclo salvar → carregar → predizer

O objetivo aqui é confirmar que o artefato é autossuficiente: outro processo deve conseguir carregá-lo e obter as mesmas predições.

### Referência

In [ ]:
import joblib

joblib.dump(model, "model.pkl")
print("Modelo salvo em model.pkl")

model_carregado = joblib.load("model.pkl")

import numpy as np
amostra = X_test[:5]
pred_original  = model.predict(amostra)
pred_carregado = model_carregado.predict(amostra)

assert np.array_equal(pred_original, pred_carregado), "Predições divergem!"
print("✅ Artefato validado — predições idênticas")
print(f"Predições: {pred_original}")

### Sua tarefa

Faça o ciclo completo:

1. salvar o modelo
2. carregar em outra variável
3. comparar predições com `assert`
4. verificar o tamanho do arquivo com `os.path.getsize("model.pkl")`

Depois, escreva 2 frases respondendo:

> O que pode acontecer se você serializar o modelo com uma versão do sklearn e tentar carregá-lo com outra?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* um arquivo `model.pkl` no diretório
* predições idênticas entre modelo original e carregado
* tamanho do arquivo medido em KB ou MB

<details>
<summary>💡 Gabarito</summary>

```python
import joblib
import numpy as np
import os

joblib.dump(model, "model.pkl")
tamanho_kb = os.path.getsize("model.pkl") / 1024
print(f"Modelo salvo: model.pkl ({tamanho_kb:.1f} KB)")

model_carregado = joblib.load("model.pkl")
amostra = X_test[:5]

pred_original  = model.predict(amostra)
pred_carregado = model_carregado.predict(amostra)

assert np.array_equal(pred_original, pred_carregado), "Predições divergem!"
print("✅ Artefato validado")
print(f"Predições: {pred_carregado}")
print(f"Probabilidades: {model_carregado.predict_proba(amostra).round(3)}")

# sklearn não garante compatibilidade irrestrita entre versões para modelos serializados.
# Por isso, em produção, é importante registrar versões exatas das dependências.
```

</details>

---

# BLOCO 3 — Hugging Face Hub como registry de modelos

**Objetivo:** publicar o artefato treinado no Hugging Face Hub com metadados mínimos, entendendo o Hub como um registry versionado de artefatos de ML.

---

## Antes de começar este bloco

Confirme que você já tem:

* uma conta no Hugging Face
* um token com permissão de escrita
* o arquivo `model.pkl` salvo localmente
* o pacote `huggingface_hub` instalado

Se algum desses itens estiver faltando, pare aqui e resolva primeiro.

---

## Conceitos-chave do Bloco 3

### O que é um model registry?

É um repositório centralizado de modelos com:

* versionamento
* metadados
* rastreabilidade

Sem um registry, modelos ficam espalhados em notebooks, pastas locais e mensagens. A consequência é simples: ninguém sabe qual versão está em produção.

### O que é um model card?

É o `README.md` do repositório do modelo. Ele documenta:

* para que o modelo serve
* com que dados foi treinado
* quais métricas alcançou
* limitações conhecidas

Um modelo sem documentação é apenas um arquivo opaco.

### Estrutura esperada do repositório no Hub

```text
seu-usuario/nome-do-modelo/
├── README.md
├── model.pkl
└── requirements.txt
```

---

## Exercício 3.1 — Autenticar e criar o repositório

**Nível:** Essencial  
**Conceito:** `huggingface_hub`, login com token, criação de repositório

### Referência

In [ ]:
from huggingface_hub import HfApi, login

login(token="hf_seu_token_aqui")

api = HfApi()

repo_url = api.create_repo(
    repo_id="seu-usuario/fraud-detector-v1",
    repo_type="model",
    exist_ok=True,
    private=False
)
print(f"Repositório: {repo_url}")

### Segurança

Nunca coloque seu token diretamente no código que irá para o repositório Git. Prefira:

* variável de ambiente
* `huggingface-cli login`

### Sua tarefa

Autentique-se e crie um repositório para o modelo do seu time.

Sugestão de convenção: `seu-usuario/mlops-[dominio]-v1`

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* autenticação válida
* nome de usuário recuperado pela API
* repositório criado ou reutilizado com `exist_ok=True`

<details>
<summary>💡 Gabarito</summary>

```python
import os
from huggingface_hub import HfApi, login

token = os.environ.get("HF_TOKEN")
login(token=token)

api = HfApi()
username = api.whoami()["name"]
print(f"Autenticado como: {username}")

repo_id = f"{username}/mlops-fraud-v1"
repo_url = api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
    private=False
)
print(f"✅ Repositório pronto: {repo_url}")
```

</details>

---

## Exercício 3.2 — Escrever o model card

**Nível:** Essencial  
**Conceito:** `README.md` como documentação do artefato

O model card deve ser criado antes do upload. Ele será a primeira coisa que alguém verá ao abrir o repositório.

### Referência — template mínimo

In [ ]:
MODEL_CARD = """---
language: pt
tags:
  - sklearn
  - classification
  - fraud-detection
  - mlops
---

# fraud-detector-v1

Modelo de classificação binária para detecção de transações fraudulentas.
Desenvolvido como parte do curso de MLOps — Semana 3.

## Uso

```python
from huggingface_hub import hf_hub_download
import joblib

model = joblib.load(hf_hub_download("seu-usuario/mlops-fraud-v1", "model.pkl"))
features = [[250.0, 14, 12.5, 1, 0]]
prediction = model.predict(features)
```

## Features de entrada

| Feature                 | Tipo  | Descrição                              |
|-------------------------|-------|----------------------------------------|
| valor_transacao         | float | Valor da transação em reais            |
| hora_transacao          | int   | Hora do dia (0-23)                     |
| distancia_ultima_compra | float | Distância geográfica em km             |
| tentativas_senha        | int   | Tentativas de senha antes da transação |
| pais_diferente          | int   | 1 se país diferente do cadastro        |

## Métricas (test set, 20% dos dados)

- **Precision (fraude):** 0.XX
- **Recall (fraude):** 0.XX
- **F1 (fraude):** 0.XX

## Dependências

- scikit-learn==X.X.X
- joblib==X.X.X

## Limitações

Modelo treinado com dados sintéticos. Não deve ser usado em produção sem
retreinamento com dados reais e validação adequada.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(MODEL_CARD)
print("README.md criado")

### Sua tarefa

Crie o `README.md` do seu modelo preenchendo:

* métricas reais do Bloco 2
* features do seu domínio
* limitações do modelo

Não deixe valores como `XX`.

Além do arquivo, escreva 1 frase respondendo:

> Por que um model card é importante mesmo em um modelo simples?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* um arquivo `README.md`
* métricas reais preenchidas
* descrição coerente das features
* uma seção de limitações

---

## Exercício 3.3 — Publicar o modelo

**Nível:** Recomendado  
**Conceito:** `upload_file`, versionamento por commit, verificação no Hub

### Referência

In [ ]:
from huggingface_hub import HfApi
import sklearn
import joblib as jl

api = HfApi()
repo_id = "seu-usuario/mlops-fraud-v1"

requirements = f"scikit-learn=={sklearn.__version__}\njoblib=={jl.__version__}\n"
with open("requirements.txt", "w") as f:
    f.write(requirements)

for filename in ["model.pkl", "README.md", "requirements.txt"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Add {filename}"
    )
    print(f"✅ {filename} publicado")

print(f"\nRepositório: https://huggingface.co/{repo_id}")

### Sua tarefa

Publique:

* `model.pkl`
* `README.md`
* `requirements.txt`

Depois, abra o repositório no navegador e confirme:

1. o README está renderizado
2. o `model.pkl` aparece na lista de arquivos
3. o `requirements.txt` está presente

Por fim, responda em 2 frases:

> O que muda se você publicar um novo `model.pkl` depois de retreinar?
> Como saber qual versão está em produção?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve ter:

* três arquivos publicados
* um histórico de commits no Hub
* URL válida do repositório do modelo

<details>
<summary>💡 Gabarito</summary>

```python
from huggingface_hub import HfApi
import sklearn
import joblib as jl
import numpy as np

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/mlops-fraud-v1"

requirements = (
    f"scikit-learn=={sklearn.__version__}\n"
    f"joblib=={jl.__version__}\n"
    f"numpy=={np.__version__}\n"
)

with open("requirements.txt", "w") as f:
    f.write(requirements)

for filename in ["model.pkl", "README.md", "requirements.txt"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"chore: add {filename}"
    )
    print(f"✅ {filename} publicado")

print(f"\n🔗 https://huggingface.co/{repo_id}")

# Cada upload gera um novo commit com hash único.
# Para rastrear a versão em produção, registre o repo_id e o commit hash
# junto com o deploy. O Hub exibe o histórico completo em
# Files and versions → History.
```

</details>

---

# BLOCO 4 — Carregar o modelo e integrar à API

**Objetivo:** criar uma função `load_model(force_download)` e integrá-la ao endpoint `/predict` da API FastAPI.

---

## Conceitos-chave do Bloco 4

### Cache local vs. download

`hf_hub_download` usa cache automaticamente:

* primeira chamada: baixa o arquivo
* chamadas seguintes: reutiliza a cópia local

Quando `force_download=True`, o cache é ignorado e o arquivo é baixado novamente.

### O modelo deve carregar uma vez

Carregar o modelo a cada requisição é um erro comum.

O padrão correto é:

* carregar o modelo uma vez na inicialização
* mantê-lo em memória
* reutilizá-lo em todos os requests

---

## Exercício 4.1 — A função `load_model`

**Nível:** Essencial  
**Conceito:** `hf_hub_download`, cache, `force_download`

### Referência

In [ ]:
from huggingface_hub import hf_hub_download
import joblib

def load_model(
    repo_id: str,
    filename: str = "model.pkl",
    force_download: bool = False
):
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    model = joblib.load(local_path)
    origem = "Hub (forçado)" if force_download else "cache local"
    print(f"✅ Modelo carregado de: {origem} ({local_path})")
    return model

### Sua tarefa

Implemente `load_model` para o seu repositório e execute:

1. primeira chamada
2. segunda chamada
3. terceira chamada com `force_download=True`

Use `time.time()` antes e depois para medir o tempo gasto em cada chamada.

Depois, escreva 2 frases respondendo:

> Por que o cache é útil?
> Em que situação `force_download=True` faria sentido?

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve observar:

* primeira chamada mais lenta
* segunda chamada mais rápida
* terceira chamada forçando novo download

<details>
<summary>💡 Gabarito</summary>

```python
from huggingface_hub import hf_hub_download
import joblib
import time

def load_model(
    repo_id: str,
    filename: str = "model.pkl",
    force_download: bool = False
):
    t0 = time.time()
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    model = joblib.load(local_path)
    elapsed = time.time() - t0
    origem = "Hub (forçado)" if force_download else "cache local"
    print(f"✅ Modelo carregado de {origem} em {elapsed:.3f}s")
    return model

REPO_ID = "seu-usuario/mlops-fraud-v1"

print("--- Primeira chamada ---")
model = load_model(REPO_ID)

print("\n--- Segunda chamada ---")
model = load_model(REPO_ID)

print("\n--- Terceira chamada (forçada) ---")
model = load_model(REPO_ID, force_download=True)
```

</details>

---

## Exercício 4.2 — Endpoint `/predict` com modelo real

**Nível:** Essencial  
**Conceito:** integração do modelo na API FastAPI, schema de entrada e saída

Este é o exercício central do caderno. Aqui você integra tudo o que construiu antes.

> **Atenção:** a ordem das features no `np.array` do endpoint deve ser **idêntica** à ordem das colunas usada no treino. Uma inversão silenciosa aqui não gera erro — gera predições erradas.

### Referência — `routers/predict.py`

In [ ]:
from fastapi import APIRouter
from pydantic import BaseModel, Field
import numpy as np

router = APIRouter()

REPO_ID = "seu-usuario/mlops-fraud-v1"
_model = None

def get_model():
    global _model
    if _model is None:
        from model_utils import load_model
        _model = load_model(REPO_ID)
    return _model


class PredictInput(BaseModel):
    valor_transacao: float = Field(gt=0, description="Valor em reais")
    hora_transacao: int = Field(ge=0, le=23, description="Hora do dia")
    distancia_ultima_compra: float = Field(ge=0, description="Distância em km")
    tentativas_senha: int = Field(ge=1, description="Tentativas de senha")
    pais_diferente: int = Field(ge=0, le=1, description="1 se país diferente")


class PredictOutput(BaseModel):
    prediction: int
    probability: float
    label: str
    model_version: str


@router.post("/predict", response_model=PredictOutput)
async def predict(input: PredictInput):
    model = get_model()

    features = np.array([[
        input.valor_transacao,
        input.hora_transacao,
        input.distancia_ultima_compra,
        input.tentativas_senha,
        input.pais_diferente
    ]])

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])
    label = "fraude" if prediction == 1 else "legítimo"

    return PredictOutput(
        prediction=prediction,
        probability=round(probability, 4),
        label=label,
        model_version=REPO_ID
    )

### Sua tarefa

1. crie `model_utils.py` com a função `load_model`
2. crie `routers/predict.py` adaptado ao seu domínio
3. registre o router no `main.py` com `prefix="/ml"` e tag `"ML"`  
   *(adapte para os routers que você já tem da semana anterior)*
4. teste no Swagger com um exemplo que pareça positivo e um que pareça negativo

Depois, escreva 2 frases avaliando:

> As probabilidades retornadas pareceram intuitivas?
> O endpoint ficou coerente com o domínio escolhido?

In [ ]:
# ✏️ Implemente no seu ambiente local


### Checkpoint

Se deu certo até aqui, você deve conseguir:

* abrir o Swagger
* ver a rota `POST /ml/predict`
* enviar um payload válido
* receber `prediction`, `probability`, `label` e `model_version`

<details>
<summary>💡 Gabarito — main.py e model_utils.py</summary>

```python
# main.py — adicione ao existente (adapte para os routers que você já tem)
from routers import predict
app.include_router(predict.router, prefix="/ml", tags=["ML"])
```

```python
# model_utils.py
from huggingface_hub import hf_hub_download
import joblib

def load_model(
    repo_id: str,
    filename: str = "model.pkl",
    force_download: bool = False
):
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    return joblib.load(local_path)
```

</details>

---

## Exercício 4.3 — Endpoint `/health` com status do modelo

**Nível:** Recomendado  
**Conceito:** distinguir "API no ar" de "modelo disponível"

Um `/health` que só retorna `{"status": "ok"}` informa apenas que o processo está vivo. Isso não basta.

Para uma API de ML, o health check precisa informar se o modelo está de fato carregável e utilizável.

### Referência

In [ ]:
@router.get("/health")
async def health():
    try:
        model = get_model()
        test_input = np.zeros((1, 5))
        model.predict(test_input)
        model_status = "ok"
        model_info = REPO_ID
    except Exception as e:
        model_status = "degraded"
        model_info = str(e)

    return {
        "api": "ok",
        "model": model_status,
        "model_repo": model_info
    }

### Sua tarefa

Implemente `/health` no seu router e teste dois cenários:

1. `REPO_ID` correto → `"model": "ok"`
2. `REPO_ID` inválido propositalmente → `"model": "degraded"`

Depois, decida e justifique em comentário:

* retornar `200` quando o modelo está degradado
* ou retornar `503`

In [ ]:
# ✏️ Escreva sua solução aqui


### Checkpoint

Se deu certo até aqui, você deve conseguir:

* chamar `GET /ml/health`
* observar diferença entre modelo disponível e indisponível
* justificar o status HTTP escolhido

<details>
<summary>💡 Gabarito</summary>

```python
from fastapi.responses import JSONResponse
import numpy as np

@router.get("/health")
async def health():
    try:
        model = get_model()
        test_input = np.zeros((1, 5))
        model.predict(test_input)
        model_ok = True
        model_info = REPO_ID
        detail = None
    except Exception as e:
        model_ok = False
        model_info = REPO_ID
        detail = str(e)

    body = {
        "api": "ok",
        "model": "ok" if model_ok else "degraded",
        "model_repo": model_info,
        "detail": detail
    }

    # 503 quando o modelo está degradado: a API está no ar,
    # mas não consegue cumprir sua função principal.
    # Retornar 200 aqui enganaria load balancers e ferramentas
    # de monitoramento que dependem do status HTTP para rotear tráfego.
    status_code = 200 if model_ok else 503
    return JSONResponse(content=body, status_code=status_code)
```

</details>

---

# Mapa do que foi construído

| Componente       | Arquivo                | O que faz                              |
| ---------------- | ---------------------- | -------------------------------------- |
| Gerador de dados | `data_utils.py`        | Produz dataset sintético reproduzível  |
| Treinamento      | notebook ou `train.py` | Treina e serializa `model.pkl`         |
| Registry         | Hugging Face Hub       | Armazena e versiona o artefato         |
| Loader           | `model_utils.py`       | `load_model(force_download)` com cache |
| API — predição   | `routers/predict.py`   | `POST /ml/predict`                     |
| API — saúde      | `routers/predict.py`   | `GET /ml/health`                       |

---

# Erros comuns neste caderno

Antes de encerrar, vale registrar os erros que mais costumam travar esse tipo de atividade:

1. **token do Hugging Face exposto no código**
2. **`repo_id` errado**
3. **modelo salvo com nome diferente de `model.pkl`**
4. **ordem das features no endpoint diferente da ordem usada no treino**
5. **import quebrado entre `main.py`, `model_utils.py` e `routers/predict.py`**
6. **modelo carregado a cada request em vez de uma vez só**
7. **ambiente de inferência com versão incompatível do sklearn**

Se algo falhar, revise esses pontos primeiro.

---

# Checklist de competências

Marque o que você consegue fazer ao final deste caderno.

## Bloco 1 — Dados sintéticos

* [ ] Gerar datasets com `make_classification` e parâmetros controlados
* [ ] Criar features nomeadas e coerentes com um domínio de negócio
* [ ] Escrever uma função geradora reproduzível com seed

## Bloco 2 — Treinamento e serialização

* [ ] Treinar um `RandomForestClassifier` e interpretar `classification_report`
* [ ] Serializar e desserializar um modelo com `joblib`
* [ ] Validar que o artefato produz predições idênticas ao modelo original

## Bloco 3 — Hugging Face Hub

* [ ] Autenticar com token sem expor credenciais no código
* [ ] Criar repositório e publicar artefatos com mensagem de commit
* [ ] Escrever um model card com métricas reais e documentação de features

## Bloco 4 — Integração com a API

* [ ] Implementar `load_model(force_download)` com cache automático
* [ ] Carregar o modelo uma única vez na inicialização da aplicação
* [ ] Criar `POST /predict` com schema Pydantic tipado
* [ ] Criar `GET /health` que distingue API no ar de modelo disponível

---

# Conexão com a Semana 3 — CI com GitHub Actions

Na próxima sessão, vamos automatizar os testes com GitHub Actions. Isso levanta uma questão que vale discutir com o time antes da aula:

> O modelo `.pkl` deve estar commitado no repositório Git, ou o pipeline de CI deve baixá-lo do Hugging Face Hub antes de rodar os testes?

Cada abordagem tem trade-offs. Pense nisso e traga sua opinião — será o ponto de partida da discussão na semana 3.